# Direct questioning — Gemma 4 31B on StrategyClaim (500 claims)

The model is asked whether the claim is true or false (temperature 0), without and with the annotators' facts. Results: 77.8 without facts, 86.2 with facts.

Run on Kaggle; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`).

In [2]:
!pip install -q openai datasets scikit-learn

import os, re, sys, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk

key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
client = OpenAI(api_key=key,
                base_url="https://openrouter.ai/api/v1",
                timeout=180)
MODEL_NAME = "google/gemma-4-31b-it"
WORK_DIR = Path("/kaggle/working/gemma_direct")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Gemma models on OpenRouter:")
for m in client.models.list().data:
    if "gemma" in m.id.lower():
        print("  ", m.id)


def strip_thought(text):
    text = text or ""
    text = re.sub(r"<thought>.*?</thought>", "", text, flags=re.S)
    text = re.sub(r"^.*?</thought>", "", text, flags=re.S)
    return text.strip()


def call(prompt, temperature, max_tokens):
    last = None
    for a in range(20):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, temperature=temperature,
                max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}])
            t = strip_thought(r.choices[0].message.content)
            if t:
                return t
            raise ValueError("empty")
        except Exception as e:
            last = e
            if "402" in str(e):
                raise RuntimeError("no OpenRouter credit left")
            if "404" in str(e):
                raise RuntimeError("wrong MODEL_NAME, see list above")
            if "429" in str(e) or "rate" in str(e).lower():
                time.sleep(30)
            else:
                time.sleep(5)
    raise RuntimeError(f"failed: {last}")


print("test:", call("Is Paris in France? Answer true or false.",
                    0, 1500))

Gemma models on OpenRouter:
   google/gemma-4-26b-a4b-it
   google/gemma-4-26b-a4b-it:free
   google/gemma-4-31b-it
   google/gemma-4-31b-it:free
   google/gemma-3-4b-it
   google/gemma-3-12b-it
   google/gemma-3-27b-it
   google/gemma-2-27b-it
test: True


In [3]:
repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q "
              "https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk(repo + "/Datasets/StrategyQA/Experiment")
df = df.to_pandas().reset_index(drop=True)


def format_facts(f):
    if isinstance(f, (list, tuple, np.ndarray)):
        return "\n".join("- " + str(x) for x in f)
    return "" if f is None else str(f)


claims = df["claim"].astype(str).tolist()
facts = df["facts"].apply(format_facts).tolist()
gold = df["valid"].apply(lambda v: int(bool(v))).tolist()
print("claims:", len(claims), "| true:", sum(gold))

claims: 500 | true: 250


In [4]:
import os, time, pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

CK = WORK_DIR / "gemma_direct_500.pkl"
WORKERS = 4


def parse_tf(text):
    t = (text or "").strip().lower().lstrip("*\"' ")
    if t.startswith("true"):
        return 1
    if t.startswith("false"):
        return 0
    has_t, has_f = "true" in t, "false" in t
    if has_t and not has_f:
        return 1
    if has_f and not has_t:
        return 0
    return None


def ask(claim, fact=None):
    if fact:
        p = (f"Facts:\n{fact}\n\nUsing these facts, is the following "
             'statement true or false? Answer with only one word: '
             f'"true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    else:
        p = ('Is the following statement true or false? Answer with '
             'only one word: "true" or "false".\n\n'
             f'Statement: {claim}\n\nAnswer:')
    return parse_tf(call(p, 0, 1500))


def run_one(i):
    return {"gold": gold[i],
            "no_facts": ask(claims[i]),
            "with_facts": ask(claims[i], facts[i])}


def load_c():
    return pickle.load(open(CK, "rb")) if CK.exists() else {}


def save_c(d):
    tmp = CK.with_suffix(".tmp")
    pickle.dump(d, open(tmp, "wb"))
    tmp.replace(CK)


def safe_one(i):
    try:
        return i, run_one(i), None
    except Exception as e:
        return i, None, str(e)[:80]


def results(d):
    idx = sorted(d)
    g = [d[i]["gold"] for i in idx]
    rows = []
    for name, k in [("Direct (no facts)", "no_facts"),
                    ("Direct (with facts)", "with_facts")]:
        raw = [d[i][k] for i in idx]
        p = [x if x is not None else 0 for x in raw]
        tn, fp, fn, tp = confusion_matrix(g, p, labels=[0, 1]).ravel()
        rows.append({"method": name, "n": len(p),
                     "acc": round(accuracy_score(g, p) * 100, 1),
                     "prec": round(precision_score(
                         g, p, zero_division=0) * 100, 1),
                     "rec": round(recall_score(
                         g, p, zero_division=0) * 100, 1),
                     "f1": round(f1_score(
                         g, p, zero_division=0) * 100, 1),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                     "unparsed": sum(x is None for x in raw)})
    print(pd.DataFrame(rows).to_string(index=False), flush=True)


def run_pass(label):
    d = load_c()
    todo = [i for i in range(len(claims)) if i not in d]
    print(f"[{label}] have {len(d)}/500, running {len(todo)}",
          flush=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = [pool.submit(safe_one, i) for i in todo]
        for f in as_completed(futs):
            i, v, err = f.result()
            if err:
                print(f"  error at {i}: {err}", flush=True)
                continue
            d[i] = v
            save_c(d)
            if len(d) % 50 == 0:
                mins = (time.time() - t0) / 60
                print(f"--- {len(d)}/500 | {mins:.0f} min ---")
                results(d)
    return load_c()


d = run_pass("main run")
for r in range(2):
    if len(d) < len(claims):
        time.sleep(30)
        d = run_pass(f"retry {r + 1}")

os.chdir("/kaggle/working")
rows = [dict(d[i], index=i) for i in sorted(d)]
pd.DataFrame(rows).to_csv("gemma_direct_500.csv", index=False)

print(f"\n=== FINAL: Gemma 4 31B - StrategyClaim - Direct "
      f"({len(d)}/500) ===")
results(d)

[main run] have 0/500, running 500
--- 50/500 | 0 min ---
             method  n  acc  prec  rec   f1  TN  FP  FN  TP  unparsed
  Direct (no facts) 50 72.0  77.3 65.4 70.8  19   5   9  17         0
Direct (with facts) 50 82.0 100.0 65.4 79.1  24   0   9  17         0
--- 100/500 | 0 min ---
             method   n  acc  prec  rec   f1  TN  FP  FN  TP  unparsed
  Direct (no facts) 100 78.0  84.4 71.7 77.6  40   7  15  38         0
Direct (with facts) 100 85.0 100.0 71.7 83.5  47   0  15  38         0
--- 150/500 | 1 min ---
             method   n  acc  prec  rec   f1  TN  FP  FN  TP  unparsed
  Direct (no facts) 150 78.7  80.0 73.2 76.5  66  13  19  52         0
Direct (with facts) 150 88.0  98.2 76.1 85.7  78   1  17  54         0
--- 200/500 | 1 min ---
             method   n  acc  prec  rec   f1  TN  FP  FN  TP  unparsed
  Direct (no facts) 200 79.0  79.5 74.5 76.9  88  18  24  70         0
Direct (with facts) 200 89.0  97.4 78.7 87.1 104   2  20  74         0
--- 250/500 | 1 min -